# Exploratory Data Analysis, Feature Engineering & Unsupervised Learning

## Project
**Mathematical Fingerprint of Musical Success**

## Purpose
This notebook focuses on exploring the cleaned datasets produced in ***math_music_data_cleaning.ipynb*** and preparing them for machine learning tasks.

The notebook covers:
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Dimensionality Reduction (PCA)
- Clustering (K-Means)
- Time Series Analysis

---

## Data Setup (Required Before Running the Notebook)

### Step 1: Create the Data Folder
1. Open the **Files** panel (folder icon on the left side of Colab).
2. Create a folder named:

```text
data
```

### Step 2: Upload the Processed Datasets
Upload the following files generated in Notebook 1 into the `data/` folder:

```text
data/master_dataset_clean.csv
data/corgis_historical_clean.csv
```

### Step 3: Verify the Files
Run **CELL 1 – File Check** before continuing.

Expected structure:

```text
content/
│
├── data/
│   ├── master_dataset_clean.csv
│   └── corgis_historical_clean.csv
│
└── sample_data/
```

---

## Input Datasets

### master_dataset_clean.csv
Contains:
- Spotify audio features
- Billboard hit information
- Target variable (`is_hit`)

### corgis_historical_clean.csv
Contains:
- Historical music information
- Year-related variables
- Metadata used for time-series analysis

---

## Expected Outputs

By the end of this notebook we will produce:

- Correlation analysis
- Engineered musical features
- PCA components
- K-Means clusters
- Time-series visualizations
- Processed dataset for ***math_music_modeling_and_mlflow.ipynb***

---

In [ ]:
# ============================================================
# Topic: File Check
# Goal: Verify that files exist inside the 'data' folder
# ============================================================

import os

required_files = [
    "data/master_dataset_clean.csv",
    "data/corgis_historical_clean.csv"
]

all_ok = True
for f in required_files:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"file: {f} — {size_mb:.2f} MB")
    else:
        print(f" {f} — NOT FOUND in 'data' folder.")
        all_ok = False

if all_ok:
    print("\n Validation successful. Continuing ...")

In [ ]:
# ============================================================
# Topic: Data Loading
# Goal: Load the cleaned datasets produced in math_music_data_cleaning.ipynb
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# ------------------------------------------------------------
# Load cleaned datasets
# ------------------------------------------------------------

df = pd.read_csv("data/master_dataset_clean.csv")
df_hist = pd.read_csv("data/corgis_historical_clean.csv")

# ------------------------------------------------------------
# Audio features used throughout the project
# ------------------------------------------------------------

audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

print("Datasets loaded successfully")
print(f"Master dataset shape: {df.shape}")
print(f"Historical dataset shape: {df_hist.shape}")

display(df.head())

In [ ]:
# ============================================================
# Topic: Exploratory Data Analysis (EDA)
# Goal: Inspect dataset structure and available features
# ============================================================

print("=" * 60)
print("MASTER DATASET OVERVIEW")
print("=" * 60)

print("\nDataset shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nData types:")
display(df.dtypes)

print("\nFirst statistics for numerical columns:")
display(df.describe())

In [ ]:
# ============================================================
# Topic: Exploratory Data Analysis (EDA) - Correlation Analysis
# Goal: Examine relationships between numerical audio features,
#       popularity, and hit status
# ============================================================

# ------------------------------------------------------------
# Select numerical columns for correlation analysis
# NOTE:
# Exclude 'peak_pos' and 'wks_on_chart' from the correlation
# matrix used for core interpretation because they come from
# Billboard and may introduce leakage in later modeling.
# ------------------------------------------------------------

corr_features = [
    "popularity",
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "is_hit"
]

corr_matrix = df[corr_features].corr()

print("Correlation matrix:")
display(corr_matrix)

# ------------------------------------------------------------
# Visualize correlation matrix
# ------------------------------------------------------------

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Matrix of Audio Features, Popularity, and Hit Status")
plt.show()

### Observations: Correlation Analysis

Based on the generated Heatmap, several key mathematical and musical relationships can be identified:

*   **Energy and Loudness (0.76):** There is a strong positive correlation. This confirms that songs with higher physical volume (loudness) are mathematically perceived as more energetic.
*   **Energy and Acousticness (-0.73):** A strong negative correlation is observed. This suggests that as a song becomes more acoustic, its energy level significantly drops, reflecting the difference between electronic and acoustic instrumentation.
*   **Danceability and Valence (0.49):** There is a moderate positive relationship. This implies that "happier" or more positive songs (high valence) tend to be more danceable, which is a common trend in commercial music.
*   **Hit Status vs. Popularity (0.13):** The correlation is surprisingly low. This is a **non-trivial finding**: it suggests that being a "Billboard Hit" historically does not always translate to high "Popularity" on Spotify today, indicating a "popularity drift" over time.
*   **Instrumentalness and Loudness (-0.43):** A negative correlation, showing that instrumental tracks (songs without vocals) are generally mixed at lower loudness levels compared to vocal-heavy tracks.

---
**ML Observation:** The analysis shows that some features, such as Energy and Loudness, contain very similar information. To reduce redundancy and make the dataset easier to work with, we will apply Principal Component Analysis (PCA) in the next stage. This technique combines related features into a smaller set of variables while preserving most of the important information.